# ❄️ Phase 2: Frost Ground Truth Auto-Annotation (Colab)

This notebook generates **automatic frost annotation masks** by subtracting the base (unfrosted) reference image `0.png` from every frosted image in a session.

### Instructions:
1. Ensure your `Frost_images` folder (extracted from the zip) is uploaded to your Google Drive.
2. Run the cell below to mount your Google Drive.
3. Run the processing cell to generate the masks.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# WARNING: Update this path to where your Frost_images folder is located in your Drive!
DATASET_PATH = "/content/drive/MyDrive/Frost_images-20260709T051618Z-3-001/Frost_images"
OUTPUT_PATH = "/content/drive/MyDrive/Frost_Annotations"

print(f"Looking for dataset at: {DATASET_PATH}")

In [ ]:
import cv2
import numpy as np
import glob
import matplotlib.pyplot as plt

EXCLUDE_FOLDERS = ["13-sep-2022", "Labeledf_8-mar-2023"]

def process_folder(folder_path, output_base_dir):
    folder_name = os.path.basename(folder_path)
    if folder_name in EXCLUDE_FOLDERS:
        print(f"Skipping {folder_name} (excluded)")
        return

    ref_paths = glob.glob(os.path.join(folder_path, "0.*"))
    if not ref_paths:
        print(f"No reference image found in {folder_name}")
        return
    
    ref_path = ref_paths[0]
    ref_img = cv2.imread(ref_path)
    if ref_img is None: return
    ref_gray = cv2.cvtColor(ref_img, cv2.COLOR_BGR2GRAY)
    
    out_dir = os.path.join(output_base_dir, folder_name)
    os.makedirs(out_dir, exist_ok=True)
    
    image_paths = glob.glob(os.path.join(folder_path, "*.*"))
    count = 0
    for img_path in image_paths:
        if img_path == ref_path: continue
            
        img_name = os.path.basename(img_path)
        img = cv2.imread(img_path)
        if img is None: continue
            
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(gray, ref_gray)
        
        # Frost threshold (adjustable)
        _, thresh = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
        
        kernel = np.ones((3,3), np.uint8)
        mask = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
        
        out_path = os.path.join(out_dir, f"{os.path.splitext(img_name)[0]}_mask.png")
        cv2.imwrite(out_path, mask)
        count += 1
        
    print(f"Processed {folder_name}: generated {count} masks.")

# Create Output Directory
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Iterate all folders
folders = [f.path for f in os.scandir(DATASET_PATH) if f.is_dir()]
print(f"Found {len(folders)} session folders. Processing...")

for folder in folders:
    process_folder(folder, OUTPUT_PATH)

print("✅ Auto-annotation complete! Masks saved to Google Drive.")